# Assignment 3: Evaluate a small RAG system (City Library FAQ)

**Submit code and generated results only. No theory answers or written report.**

The setup cell includes `ASSIGNMENT_RAG` with 3 library documents (`documents`) and 8 questions (`queries`).
Each query has: `query`, `doc_id`, `answer_span` (the exact words that answer it), `reference_answer`, and `answer_keywords` (any one of them in an answer means it is correct).

1. **Chunks and relevance.** Write `make_chunks(docs, size, overlap)` (count words, not tokens) and `grade(chunk, query)`: return 2 if the chunk contains the `answer_span` (ignore upper/lower case and extra spaces), 1 if it is from the same `doc_id` but has no answer span, otherwise 0. Check that every `answer_span` appears in its own document. Print how many chunks you get with `size=30, overlap=0`.
2. **Retrieval metrics.** Write `hit_at_k`, `reciprocal_rank`, and `ndcg_at_k` for a list of grades in ranked order (a grade of 2 or more is relevant; use `k=3`). Test them on `[0, 2, 0, 1]`. Embed the chunks and queries with `embed()` from the setup cell above, rank chunks by cosine similarity, and print the average Hit@3, MRR@3, and NDCG@3 for `size=30, overlap=0`.
3. **Grid search.** Try `size` in `[30, 60]` and `overlap` in `[0, 15]`. Print a pandas table with the number of chunks and the three metrics for each setting, and print the best setting (highest Hit@3, ties broken by MRR@3).
4. **Generation.** With the best setting, retrieve the top 3 chunks for each query and ask Gemini to answer using only those chunks. Add these columns to a pandas table: `retrieval_hit` (is the answer span in the top 3?), `answer_correct` (code check with `answer_keywords`), and `faithful` (LLM judge returning JSON with `answer` Pass/Fail and `reasoning`). Add one extra row with a **clearly marked fixture**: an intentionally unfaithful answer that you write yourself. Print the table and the counts of each diagnosis: `OK`, `generation problem` (hit but wrong), `retrieval problem` (no hit and wrong), `lucky` (no hit but correct).
5. **Context swap.** Pick one query. Answer it with its real document, then with an unrelated document, using (a) a plain system prompt and (b) a system prompt that says to use only the documents and to say "I don't know" otherwise. Print the 4 answers side by side.

**Hints:**
- Reuse `ask()` and `embed()` from the setup cell above, and ideas from Notebooks 2, 3, 5 and 7 (Chapter 7 notebook).
- Normalize text with `" ".join(text.lower().split())` before checking `answer_span`.
- Embed all texts in one `embed(list_of_texts)` call; make unit-length vectors so that cosine similarity is a dot product.
- `ndcg_at_k`: DCG = sum of `grade / log2(rank + 1)` over the top k; the ideal DCG uses the same top-k grades sorted from best to worst; return 0 if the ideal DCG is 0. For `[0, 2, 0, 1]` with `k=3` you should get RR = 0.5 and NDCG about 0.63.
- Do not combine `temperature` with `as_json=True`; use `temperature=0` only for the answering call.
- Validate the judge output: `answer` must be exactly `Pass` or `Fail`.

**Deliverables:** your solution notebook and the printed results. Keep the API key in Colab Secrets (or the root `.env` locally).

## Run in Google Colab or locally
Upload this notebook alone and run cells from top to bottom. Setup installs missing packages;
all required sample data and helper functions are included below.
For Gemini calls, add `GEMINI_API_KEY` in Colab's key-shaped **Secrets** panel and enable notebook access,
or enter your key at the hidden prompt. Local runs use the single repository-root `.env`.
Never paste a key into a code cell. API calls use your own quota; notebook 04 needs no API key.
Students: **File → Save a copy in Drive** for your own copy and session.
Generated files live in the current session; download any results you want to keep before it ends.


In [ ]:
# Run once per fresh Colab session. Local users can run uv sync at the repo root.
import importlib.util
import subprocess
import sys

packages = {"google.genai": "google-genai", "dotenv": "python-dotenv",
            "numpy": "numpy", "pandas": "pandas", "sklearn": "scikit-learn",
            "rank_bm25": "rank-bm25"}
def available(module):
    try:
        return importlib.util.find_spec(module) is not None
    except ModuleNotFoundError:
        return False
missing = [package for module, package in packages.items() if not available(module)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])


In [ ]:
"""Small helpers shared by all notebooks (Gemini calls + data loading)."""
import json
import os
import time
from pathlib import Path

import numpy as np
from dotenv import load_dotenv
from google import genai
from google.genai import types

# Colab uses its own Secrets; local runs find the root .env.
try:
    from google.colab import userdata
except ImportError:
    userdata = None
if userdata is None:
    for directory in (Path.cwd(), *Path.cwd().parents):
        if (directory / ".env").is_file():
            load_dotenv(directory / ".env", override=False)
            break

MODEL = os.getenv("GEMINI_MODEL", "gemini-2.5-flash")
EMBED_MODEL = os.getenv("GEMINI_EMBED_MODEL", "gemini-embedding-001")

_client = None


def client():
    """Create the Gemini client once and reuse it."""
    global _client
    if _client is None:
        key = os.getenv("GEMINI_API_KEY")
        if not key and userdata is not None:
            try:
                key = userdata.get("GEMINI_API_KEY")
            except Exception:
                pass
        if not key:
            from getpass import getpass
            key = getpass("Enter your own Gemini API key (hidden): ").strip()
        if not key:
            raise RuntimeError("GEMINI_API_KEY is missing. Add it to Colab Secrets or the root .env.")
        _client = genai.Client(api_key=key)
    return _client



def _generate(contents, system=None, temperature=0.0, as_json=False, model=None, retries=5):
    config = types.GenerateContentConfig(
        system_instruction=system,
        temperature=temperature,
        response_mime_type="application/json" if as_json else None,
    )
    for attempt in range(retries):
        try:
            resp = client().models.generate_content(model=model or MODEL, contents=contents, config=config)
            return json.loads(resp.text) if as_json else resp.text
        except Exception as e:  # rate limits (free tier) or temporary errors
            if attempt == retries - 1:
                raise
            wait = 10 * (attempt + 1)
            print(f"[retry {attempt + 1}] {type(e).__name__}: waiting {wait}s")
            time.sleep(wait)


def ask(prompt, system=None, temperature=0.0, as_json=False, model=None):
    """One prompt in, one answer out. as_json=True returns a Python dict."""
    return _generate(prompt, system, temperature, as_json, model)


def chat(history, system=None, temperature=0.0, model=None):
    """history = [{"role": "user" | "assistant", "content": "..."}, ...]"""
    contents = [
        types.Content(role="model" if m["role"] == "assistant" else "user",
                      parts=[types.Part(text=m["content"])])
        for m in history
    ]
    return _generate(contents, system, temperature, False, model)


def embed(texts):
    """Turn a list of texts into a numpy matrix (one row per text)."""
    r = client().models.embed_content(model=EMBED_MODEL, contents=texts)
    return np.array([e.values for e in r.embeddings])


def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


# All sample prompts and data are embedded in this notebook.
def load_data():
    return {}

ASSIGNMENT_RAG = {'documents': [{'id': 'borrowing', 'text': 'The City Library lets members borrow up to 10 books at a time for a period of 21 days. Books can be renewed online two times, unless another member has reserved them. A late fee of $0.25 per day is charged for each overdue book, and the fee stops growing once it reaches $10. Members who lose a book must pay its replacement cost plus a $5 processing fee. Membership is free for residents of the city, while visitors pay $15 per year for a card. DVDs can be borrowed for 7 days only.'}, {'id': 'hours_rooms', 'text': 'The main library is open Monday to Thursday from 8 AM to 8 PM, and on Friday from 8 AM to 5 PM. On Saturdays the library opens at 10 AM and closes at 4 PM, and it is closed on Sundays and public holidays. Study rooms can be reserved up to 14 days in advance for a maximum of 3 hours per day. Each study room fits up to 6 people and has a whiteboard. Group reservations need a library card from at least one member. Printing costs $0.10 per page in black and white and $0.50 per page in color. Free wifi is available in the whole building.'}, {'id': 'events', 'text': "The library runs a children's story hour every Wednesday at 4 PM in the reading corner. It is free and suitable for ages 3 to 7. The summer reading challenge starts on June 1 and ends on August 31, and children who read 12 books receive a free ticket to the science museum. Adult book clubs meet on the last Thursday of each month, and the next book is announced two weeks before. The library also offers a free computer class for seniors on Tuesday mornings at 10 AM. Registration for all events is done at the front desk or on the library website."}], 'queries': [{'query': 'How many books can I take out at once?', 'doc_id': 'borrowing', 'answer_span': '10 books', 'reference_answer': 'Up to 10 books at a time.', 'answer_keywords': ['10']}, {'query': 'What is the late fee for an overdue book?', 'doc_id': 'borrowing', 'answer_span': '$0.25 per day', 'reference_answer': '$0.25 per day per overdue book.', 'answer_keywords': ['0.25', '25 cents']}, {'query': 'How much does a library card cost if I live outside the city?', 'doc_id': 'borrowing', 'answer_span': '$15 per year', 'reference_answer': 'Visitors pay $15 per year.', 'answer_keywords': ['15']}, {'query': 'Can I visit the library on Sunday?', 'doc_id': 'hours_rooms', 'answer_span': 'closed on Sundays', 'reference_answer': 'No, the library is closed on Sundays.', 'answer_keywords': ['closed']}, {'query': 'How long can I book a study room each day?', 'doc_id': 'hours_rooms', 'answer_span': '3 hours per day', 'reference_answer': 'A maximum of 3 hours per day.', 'answer_keywords': ['3 hours', 'three hours']}, {'query': 'What is the price to print in color?', 'doc_id': 'hours_rooms', 'answer_span': '$0.50 per page in color', 'reference_answer': '$0.50 per page.', 'answer_keywords': ['0.50', '50 cents']}, {'query': 'When is story time for kids?', 'doc_id': 'events', 'answer_span': 'every Wednesday at 4 PM', 'reference_answer': 'Every Wednesday at 4 PM.', 'answer_keywords': ['wednesday']}, {'query': 'What do children get for finishing the summer reading challenge?', 'doc_id': 'events', 'answer_span': '12 books', 'reference_answer': 'A free ticket to the science museum.', 'answer_keywords': ['museum']}]}


In [ ]:
# Starter cell: add your solution cells below this one.
import json, math, re
import numpy as np
import pandas as pd
from IPython.display import display

data = ASSIGNMENT_RAG
docs, queries = data["documents"], data["queries"]
K = 3
print("Model:", MODEL, "|", len(docs), "documents |", len(queries), "queries")